In [49]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [50]:
train_data = pd.read_csv("samsum-train.csv")
val_data   = pd.read_csv("samsum-validation.csv")

In [51]:
train_data.head()
val_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


# DATA preprocessing

In [52]:
import re

def clean_data(data):
    data = re.sub(r"\r\n", " ", data)
    data = re.sub(r"\s+", " ", data)
    data = re.sub(r"<.*?>", " ", data)
    data = data.strip().lower()
    return data

In [53]:
train_data = train_data.dropna(subset=["dialogue", "summary"])
val_data = val_data.dropna(subset=["dialogue", "summary"])

train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"]  = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

# Tokenize the data

In [54]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [55]:
length = []

for dialogue in train_data["dialogue"]:
    token_len = len(tokenizer(dialogue).input_ids)
    length.append(token_len)

print(max(length))
print(sum(length) / len(length))

cnt = sum(1 for i in length if i > 512)

print(cnt)
print(cnt / len(length) * 100)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (594 > 512). Running this sequence through the model will result in indexing errors


1224
168.11479193537437
258
1.751408594121241


In [56]:
def tokenize(data):
    inputs  = tokenizer(
        data["dialogue"],
        padding = "max_length",
        max_length = 512,
        truncation = True
    )
    
    targets = tokenizer(
        data["summary"],
        padding = "max_length",
        max_length = 150,
        truncation = True
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

In [57]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset   = val_data.apply(tokenize, axis = 1).tolist()

# Generation Task

In [58]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [59]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [60]:
# training args

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 6,
    weight_decay = 0.01,
    per_device_eval_batch_size = 16,
    per_device_train_batch_size = 16,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    warmup_steps = 500
)

In [61]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset
)

In [62]:
# trainer.train()

In [67]:
# model.save_pretrained("final_model")
# tokenizer.save_pretrained("final_model")

In [64]:
# !zip -r final_model.zip final_model

In [65]:
# from google.colab import files

# files.download("final_model.zip")

In [68]:
model = T5ForConditionalGeneration.from_pretrained("./final_model")
tokenizer = T5Tokenizer.from_pretrained("./final_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [69]:
def summarize_dialogue(dialouge):
    dialouge = clean_data(dialouge)

    # tokenize
    inputs = tokenizer(
        dialouge,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt",
    )

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4,
        early_stopping = True,
    )

    # token ids convert to summary => decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens = True) # EOS, SEP
    return summary

In [70]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary : ", summary)

Summary :  experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact. ensuring fairness and transparency is becoming a key area of research.
